<a href="https://colab.research.google.com/github/akshay-aiml/LangChain_LangGraph/blob/main/RAG_Multi_Query_Rewriting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
!pip install langchain-groq langgraph  langchain-community  faiss-cpu   tiktoken -qU langchain-text-splitters langchain-huggingface


In [13]:
from google.colab import userdata

api_key1 = userdata.get("GROQ_API_KEY")

In [14]:
from langchain_core.documents import Document
# Sample documents — our 'knowledge base'
docs = [
    Document(page_content="""
    Overfitting occurs when a machine learning model learns the training data too well,
    including its noise and outliers. The model performs excellently on training data
    but poorly on unseen test data. Common solutions include regularization (L1/L2),
    dropout, early stopping, and using more training data.
    """, metadata={"source": "ml_basics", "topic": "overfitting"}),

    Document(page_content="""
    Gradient Descent is an optimization algorithm used to minimize the loss function
    in machine learning. It works by iteratively moving in the direction of the negative
    gradient. Variants include Stochastic Gradient Descent (SGD), Mini-batch GD,
    Adam optimizer, and RMSprop. Learning rate is a critical hyperparameter.
    """, metadata={"source": "ml_basics", "topic": "optimization"}),

    Document(page_content="""
    Transformers are a deep learning architecture based on self-attention mechanisms.
    Introduced in the paper 'Attention is All You Need' (2017), they replaced RNNs
    for NLP tasks. Key components: multi-head attention, positional encoding,
    feed-forward layers, encoder-decoder structure. Used in BERT, GPT, T5.
    """, metadata={"source": "deep_learning", "topic": "transformers"}),

    Document(page_content="""
    RAG (Retrieval Augmented Generation) combines a retriever and a generator.
    It retrieves relevant documents from a knowledge base and passes them as context
    to an LLM to generate answers. This reduces hallucination and allows LLMs to
    access up-to-date or domain-specific information without retraining.
    """, metadata={"source": "llm_techniques", "topic": "rag"}),

    Document(page_content="""
    Vector databases store data as high-dimensional vectors (embeddings). They enable
    semantic similarity search using metrics like cosine similarity or dot product.
    Popular vector DBs: Pinecone, Weaviate, Chroma, FAISS, Qdrant. Essential for
    building RAG systems and semantic search applications.
    """, metadata={"source": "llm_techniques", "topic": "vector_db"}),

    Document(page_content="""
    Cross-validation is a technique to evaluate model generalization. K-Fold CV splits
    data into K subsets; the model trains on K-1 folds and tests on the remaining fold,
    repeated K times. Stratified K-Fold preserves class distribution. Helps detect
    overfitting and gives a more reliable performance estimate.
    """, metadata={"source": "ml_basics", "topic": "evaluation"}),

    Document(page_content="""
    BERT (Bidirectional Encoder Representations from Transformers) is a pre-trained
    language model by Google. It uses masked language modeling and next sentence
    prediction for pre-training. BERT is bidirectional — it reads text in both
    directions simultaneously. Used for classification, NER, QA tasks via fine-tuning.
    """, metadata={"source": "deep_learning", "topic": "bert"}),

    Document(page_content="""
    Embeddings are dense vector representations of data (text, images, etc.) in a
    continuous vector space. Semantically similar items are close together. Text
    embeddings from models like OpenAI text-embedding-ada-002 or sentence-transformers
    capture meaning. Used in semantic search, clustering, RAG, and classification.
    """, metadata={"source": "llm_techniques", "topic": "embeddings"}),
]

print(f"✅ Knowledge base created with {len(docs)} documents")

✅ Knowledge base created with 8 documents


In [15]:
from google.colab import userdata

api_key = userdata.get("HGF")

In [16]:
from langchain_groq import ChatGroq
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings

# Initialize the embedding model
embeddings = HuggingFaceEmbeddings(model_name="nomic-ai/nomic-embed-text-v1.5", model_kwargs = {'trust_remote_code': True})

# Build FAISS vector store directly from documents
vectorstore = FAISS.from_documents(docs, embeddings)

# Create a base retriever (top-3 results)
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print("✅ FAISS vector store built and retriever ready!")

✅ FAISS vector store built and retriever ready!


In [17]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=api_key,
    temperature=0
)

In [18]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Initialize LLM
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=api_key1,
    temperature=0
)

def format_docs(docs):
    """Format retrieved docs into a single context string."""
    return "\n\n".join([
        f"[Doc {i+1}] ({d.metadata.get('topic', 'N/A')}):\n{d.page_content.strip()}"
        for i, d in enumerate(docs)
    ])

# Final answer prompt
answer_prompt = ChatPromptTemplate.from_template("""
You are a helpful AI assistant. Answer the question using ONLY the context below.
If the answer is not in the context, say 'I don't have enough information.'

Context:
{context}

Question: {question}

Answer:
""")

answer_chain = answer_prompt | llm | StrOutputParser()

print("✅ LLM and answer chain ready!")

✅ LLM and answer chain ready!


In [19]:
"""
Multi-Query Rewriting
#Generate N different versions of the query, retrieve docs for each, merge (deduplicate), and answer.

#This increases recall — different phrasings capture different relevant docs.


User Query  →  [LLM]  →  Query 1, Query 2, Query 3
                              ↓        ↓        ↓
                           Docs 1   Docs 2   Docs 3
                              └────────┴────────┘
                               Merged & Deduplicated
                                       ↓
                                   LLM Answer
 """

'\nMulti-Query Rewriting\n#Generate N different versions of the query, retrieve docs for each, merge (deduplicate), and answer.\n\n#This increases recall — different phrasings capture different relevant docs.\n\n\nUser Query  →  [LLM]  →  Query 1, Query 2, Query 3\n                              ↓        ↓        ↓\n                           Docs 1   Docs 2   Docs 3\n                              └────────┴────────┘\n                               Merged & Deduplicated\n                                       ↓\n                                   LLM Answer\n '

In [20]:
from langchain_core.output_parsers import BaseOutputParser
from typing import List

# Custom parser to split numbered list into a Python list
class LineListOutputParser(BaseOutputParser[List[str]]):
    """Parse LLM output into a list of strings, one per line."""
    def parse(self, text: str) -> List[str]:
        lines = text.strip().split("\n")
        # Remove numbering like '1. ', '2. ' etc.
        cleaned = []
        for line in lines:
            line = line.strip()
            if line:
                # Strip leading numbers and punctuation
                import re
                line = re.sub(r'^\d+[.)\-]\s*', '', line)
                cleaned.append(line)
        return cleaned

In [21]:
# ─── Multi-Query Prompt ───────────────────────────────────────────────────────
multi_query_prompt = ChatPromptTemplate.from_template("""
You are an AI assistant helping to improve document retrieval.
Generate {n} different versions of the given question that capture the same intent
but use different wording, perspective, or terminology.

Original question: {question}

Output ONLY the {n} questions, one per line, numbered:
""")

multi_query_chain = multi_query_prompt | llm | LineListOutputParser()

In [22]:
def multi_query_rag(question: str, n: int = 3, verbose: bool = True) -> str:
    """
    RAG pipeline with multi-query expansion.

    Steps:
      1. Generate N query variants
      2. Retrieve docs for EACH variant
      3. Deduplicate retrieved docs
      4. Generate final answer
    """
    # Step 1: Generate multiple queries
    queries = multi_query_chain.invoke({"question": question, "n": n})

    if verbose:
        print(f"📝 Original Query: {question}")
        print(f"🔀 Generated {len(queries)} query variants:")
        for i, q in enumerate(queries, 1):
            print(f"   {i}. {q}")
        print("-" * 60)

    # Step 2: Retrieve docs for each query and deduplicate
    all_docs = []
    seen_contents = set()

    for q in queries:
        retrieved = base_retriever.invoke(q)
        for doc in retrieved:
            # Deduplicate by content hash
            content_key = doc.page_content[:100]  # first 100 chars as key
            if content_key not in seen_contents:
                seen_contents.add(content_key)
                all_docs.append(doc)

    if verbose:
        print(f"📚 Total unique docs retrieved: {len(all_docs)}")
        for d in all_docs:
            print(f"   → [{d.metadata.get('topic')}]")
        print("-" * 60)

    # Step 3: Generate answer with all unique docs as context
    context = format_docs(all_docs)
    answer = answer_chain.invoke({"context": context, "question": question})

    return answer

In [23]:
# ─── Test It ──────────────────────────────────────────────────────────────────
result = multi_query_rag("how do transformers work in NLP?")
print("\n🤖 Answer:")
print(result)

📝 Original Query: how do transformers work in NLP?
🔀 Generated 3 query variants:
   1. What is the role of transformer architecture in natural language processing applications?
   2. How do transformer models process and understand human language in NLP tasks?
   3. Can you explain the mechanism of transformer-based systems in facilitating machine understanding of text data?
------------------------------------------------------------
📚 Total unique docs retrieved: 4
   → [transformers]
   → [bert]
   → [rag]
   → [embeddings]
------------------------------------------------------------

🤖 Answer:
Transformers are a deep learning architecture based on self-attention mechanisms. They work by using key components such as multi-head attention, positional encoding, feed-forward layers, and an encoder-decoder structure. This allows them to replace RNNs for NLP tasks.
